In [1]:
import pandas as pd
import networkx as nx

def build_network_by_index(
    B_nodes_csv,
    AA_edges_csv,
    AB_edges_csv,
    has_header=True
):
    header = 0 if has_header else None
    G = nx.Graph()

    # Add Nodes
    nodes = pd.read_csv(B_nodes_csv, header=header)
    for i, row in nodes.iterrows():
        G.add_node(row.iloc[0], direction=row.iloc[1])

    # Add Eeges
    for edge_csv in [AA_edges_csv, AB_edges_csv]:
        edges = pd.read_csv(edge_csv, header=header)
        for i, row in edges.iterrows():
            G.add_edge(row.iloc[0],row.iloc[1],direction=row.iloc[2])

    # Define layer 0
    layer_0 = []
    for n in nodes.iloc[:, 0]:
        if G.degree(n) > 0:
            layer_0.append(n)
    
    return G,layer_0


In [3]:
# For each node in layer 1(d) count all edges between layer 1(d) and layer 0(d-1) --> (where d represents layer)
# For each node in layer 1(d), create two variables called pos_votes and neg_votes. 

# deletes edges based on previous layer
def filter_level(G, curr_layer, prev_layer, first,d):
    temp = 0 # DELETE LATER-----------------------
    for v in list(curr_layer):  # for every node v in the current layer
        
        pos_votes = 0
        neg_votes = 0
        pos_edges = []   # edges that voted positive
        neg_edges = []   # edges that voted negative

        for u in list(G.neighbors(v)):  # u is neighbors of v from previous layer
            if u in prev_layer:
                #if "direction" not in u and first==False:
                    #print("We will run into an error at",u)
                node_dir = float(G.nodes[u]["direction"])
                edge_dir = float(G[u][v]["direction"])
                vote = node_dir * edge_dir

                if vote > 0:
                    pos_votes += 1
                    log_file.write(f"{u} votes + \n")
                    pos_edges.append((u, v))
                else:
                    neg_votes += 1
                    log_file.write(f"{u} votes - \n")
                    neg_edges.append((u, v))

        log_file.write(f"NODE- {v},\nLAYER-{d},\nPOS_VOTES- {pos_votes},\nNEG_VOTES- {neg_votes}\n")

        if (v=="L-Lysine"):
            #print("Second TIME _-----------------------")
            print("Positive votes:",pos_votes)
            print("Negative votes:",neg_votes)
            print("Neighbors",list(G.neighbors(v)))
            #print("Second TIME _-----------------------")
        # if no votes, skip (otherwise dividing by zero)
        total = pos_votes + neg_votes
        if total == 0:
            print("ERROR")
            continue

        # set direction of v based on majority vote
        if pos_votes > neg_votes:
            G.nodes[v]["direction"] = 1
            minority_edges = neg_edges
        elif pos_votes < neg_votes:
            G.nodes[v]["direction"] = -1
            minority_edges = pos_edges
        else:
            minority_edges = pos_edges + neg_edges

        # calculate puc_value --> minority / total
        G.nodes[v]["puc_value"] = len(minority_edges) / total

        if G.nodes[v]["puc_value"] > 0.2:
            # delete all edges between v and nodes in the previous layer
            if first==True: #if layer 1, remove edge
                
                if ("direction" in G.nodes[v]):
                    del G.nodes[v]["direction"]
                for u in list(G.neighbors(v)):
                    if u in prev_layer:
                        G.remove_edge(u, v)
                        log_file.write(f"{v}-{u} edge is removed\n")
                        #print(first)
                log_file.write(f"{list(G.neighbors(v))} are neighbors of {v}\n")
            else: #all other layers, remove node
                if G.has_node(v):
                    G.remove_node(v)
                    log_file.write(f"{v} node is removed\n")
        else:
            # delete only minority-vote edges
            for a, b in minority_edges:
                G.remove_edge(a, b)
                log_file.write(f"{a} {b} edge is removed MINORITY CASE\n")
        if (v=="Dopamine"):
            print("Second TIME _-----------------------")
        log_file.write("\n\n")
    return G


# deletes edges in same level
def same_level_edges(G, curr_layer,d):
    curr_layer = set(curr_layer)  

    for u in list(curr_layer): # for u in currrent layer
        if not G.has_node(u):
            continue
            
        for v in list(G.neighbors(u)): # for v in current layer that is a neighbor of u
            if v in curr_layer and G.has_node(v):
                if "direction" not in G.nodes[u] or "direction" not in G.nodes[v]:
                    #print(u,"is given another chance")
                    continue
                    
                if (v=="L-Glutamine" or u=="L-Glutamine"):
                    print("Not being removed",u,G.nodes[u]["direction"])
                    print("Not being removed",v,G.nodes[v]["direction"])
                
                node_prod = float(G.nodes[u]["direction"]) * float(G.nodes[v]["direction"])
                edge_dir = float(G[u][v]["direction"])

                if node_prod != edge_dir:  # if product of node directions != edge direction, delete edge
                    if G.has_edge(u, v):  # avoid double-delete since undirected
                        log_file.write(f"SAME EDGE REMOVAL {v}-{u} edge is removed in layer{d}\n\n\n")
                        G.remove_edge(u, v)

    return G


In [7]:
import pandas as pd

log_file = open("log_file.txt","w")

#puc_delete_threshold=0.2
graph_A, layer_0 = build_network_by_index(
    "0. Filtered Files/cpx_nodes.csv",
    "0. Filtered Files/CSF-CSF.csv",
    "0. Filtered Files/CSF-CPX",
    has_header=True
)


layers = build_layers(graph_A, layer_0, max_depth=None)
print(graph_A)

d = 1 #Start from distance = 1
while d < len(layers):
    
    # DEBUG ------------------------------------------------------------------------------------------------------------
    print("Total layers:", len(layers))
    # Number of nodes per layer
    sum = 0
    for i, layer in enumerate(layers):
        if i>0:
            sum = sum + len(layer)
            print(f"Layer {i}: {len(layer)} nodes")
            #if (i>0):
                #print(f"Layer {i}: {layer} nodes")
    print("Total number of nodes kept",sum)
    # ------------------------------------------------------------------------------------------------------------

    print("Starting with:",d)
    prev_layer = set(layers[d - 1])   # define previous layer
    curr_layer = layers[d]   # define current layer
    
    if (d == -1):
        graph_A = filter_level(graph_A, curr_layer, prev_layer, True,d)
        print("LAYER NOT FOUND")
    else:
        graph_A = filter_level(graph_A, curr_layer, prev_layer, False,d)
        
    graph_A = same_level_edges(graph_A, curr_layer,d)
    
    layers = build_layers(graph_A, layer_0, max_depth=None)
    print("done with layer:",d,"\n")
    d += 1



rows = []
for d in range(1, len(layers)):
    for node in layers[d]:
        rows.append({
            "node": node,
            "direction": graph_A.nodes[node].get("direction"),
            "puc_value": graph_A.nodes[node].get("puc_value")
        })
    


print(len(rows))
df = pd.DataFrame(rows, columns=["node", "direction", "puc_value"])
df.to_csv("new.csv", index=False)


FileNotFoundError: [Errno 2] No such file or directory: '1. Filtered Files/cpx_nodes.csv'

In [9]:
import pandas as pd
import networkx as nx

# node_to_layer = final layers after filtering
node_to_layer = {}
for li, layer_nodes in enumerate(layers):
    for n in layer_nodes:
        node_to_layer[n] = li

# Find all nodes reachable from layer_0
reachable_from_layer0 = set()
for src in layer_0:
    if src in graph_A:
        reachable_from_layer0.update(nx.node_connected_component(graph_A, src))

# Edge pairs starting from layer 1 (same layer or adjacent layers)
edge_rows = []
for u, v in graph_A.edges():
    lu = node_to_layer.get(u)
    lv = node_to_layer.get(v)

    # Skip edges if either node is not reachable from layer_0
    if u not in reachable_from_layer0 or v not in reachable_from_layer0:
        continue

    # Skip edges involving layer_0 nodes themselves
    if lu == 0 or lv == 0:
        continue

    # Keep only same-layer or adjacent-layer edges
    if abs(lu - lv) > 1:
        continue
        
    edge_rows.append({"node1": u, "node2": v})

df_edges = pd.DataFrame(edge_rows, columns=["node1", "node2"])
df_edges.to_csv("new_feci-feci_node_pairs.csv", index=False)
print("done")
print("edges written:", len(edge_rows))


NameError: name 'layers' is not defined

In [11]:
import pandas as pd

# node_to_layer = final layers after filtering
node_to_layer = {}
for li, layer_nodes in enumerate(layers):
    for n in layer_nodes:
        node_to_layer[n] = li

# Edge pairs starting from layer 1 (same layer or adjacent layers)
edge_rows = []
for u, v in graph_A.edges():
    lu = node_to_layer.get(u)
    lv = node_to_layer.get(v)

    # Start from layer 1 only (exclude minus layer 0)
    #if not (lu == 0 and lv == 1):
    if (lu is None or lv is None):
        print(lu)
        print(lv)
        print("ERROR")
        continue

    if lu + lv != 1:
        continue
    
    edge_rows.append({"node1": u,"node2": v})

df_edges = pd.DataFrame(edge_rows, columns=["node1", "node2"])
df_edges.to_csv("new_feci-cpx_node_pairs.csv", index=False)
print("done")
print("edges written:", len(edge_rows))



NameError: name 'layers' is not defined